In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 设置随机种子以确保结果可复现
np.random.seed(42)

In [ ]:
# 实验要求1：载入手写体数字识别数据集
def load_digits_dataset(file_path):
    """
    载入MAT格式的手写体数字识别数据集
    
    参数:
    file_path - MAT文件路径
    
    返回:
    X - 特征矩阵 (5000, 400)
    y - 标签向量 (5000,)
    """
    data = loadmat(file_path)
    
    # 提取特征和标签
    X = data['X']  # 5000个样本，每个样本400维特征 (20x20像素)
    y = data['y'].flatten()  # 将标签展平为一维向量
    
    # 修正标签：MATLAB索引从1开始，我们需要0-9
    y[y == 10] = 0  # 将标签10改为0
    
    return X, y

In [ ]:
 # 1. 载入数据集
data_path = "/mnt/e/平时作业/机器学习/实验1/ex3data1.mat"  # 替换为您的实际文件路径
X, y = load_digits_dataset(data_path)
    
print("数据集信息:")
print(f"样本数量: {X.shape[0]}")
print(f"特征维度: {X.shape[1]}")
print(f"类别数量: {len(np.unique(y))}")

In [ ]:
# 实验要求2：可视化十个数据样本
def visualize_digits_samples(X, y, num_samples=10):
    """
    可视化十个数据样本，每个数字类别一个
    
    参数:
    X - 特征矩阵
    y - 标签向量
    num_samples - 要可视化的样本数量 (默认为10)
    """
    plt.figure(figsize=(12, 5))
    
    # 获取每个数字类别的一个样本
    for digit in range(10):
        # 找到该数字的第一个样本索引
        idx = np.where(y == digit)[0][0]
        
        # 提取图像数据并重塑为20x20像素
        digit_image = X[idx].reshape(20, 20).T
        
        # 创建子图
        plt.subplot(2, 5, digit+1)
        plt.imshow(digit_image, cmap='gray', interpolation='nearest')
        plt.title(f"Label: {digit}")
        plt.axis('off')
    
    plt.suptitle("Ten Handwritten Digit Samples (0-9)", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
# 实验要求3：分层抽样划分训练集和测试集
def stratified_train_test_split(X, y, test_size=0.2):
    """
    使用分层抽样划分训练集和测试集
    
    参数:
    X - 特征矩阵
    y - 标签向量
    test_size - 测试集比例 (默认0.2, 即1000个样本)
    
    返回:
    X_train, X_test, y_train, y_test
    """
    # 使用sklearn的分层抽样函数
    return train_test_split(X, y, test_size=test_size, stratify=y, random_state=42)

In [ ]:
# 2. 可视化十个数据样本（每个类别一个）
visualize_digits_samples(X, y)
    
# 3. 分层抽样划分训练集和测试集
X_train, X_test, y_train, y_test = stratified_train_test_split(X, y)
    
print("\n数据集划分结果:")
print(f"训练集大小: {X_train.shape[0]} samples")
print(f"测试集大小: {X_test.shape[0]} samples")

In [ ]:
# 实验要求4：多层感知机前馈计算
class MultiLayerPerceptron:
    """
    多层感知机(MLP)实现
    """
    def __init__(self, input_size, hidden_size, output_size):
        """
        初始化MLP参数
        
        参数:
        input_size - 输入层大小 (400)
        hidden_size - 隐藏层大小
        output_size - 输出层大小 (10)
        """
        # 初始化权重和偏置
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))
        
    def relu(self, Z):
        """ReLU激活函数"""
        return np.maximum(0, Z)
    
    def softmax(self, Z):
        """Softmax激活函数"""
        exp_Z = np.exp(Z - np.max(Z, axis=1, keepdims=True))
        return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)
    
    def forward(self, X):
        """
        前向传播计算
        
        参数:
        X - 输入特征矩阵 (m x input_size)
        
        返回:
        A2 - 输出层激活值 (softmax概率分布)
        cache - 缓存中间结果用于反向传播
        """
        # 输入层到隐藏层
        Z1 = X.dot(self.W1) + self.b1
        A1 = self.relu(Z1)
        
        # 隐藏层到输出层
        Z2 = A1.dot(self.W2) + self.b2
        A2 = self.softmax(Z2)
        
        # 缓存中间结果用于反向传播
        cache = {
            "Z1": Z1, "A1": A1,
            "Z2": Z2, "A2": A2,
            "X": X
        }
        
        return A2, cache

In [ ]:
# 实验要求5：交叉熵损失函数和梯度计算
def cross_entropy_loss(y_pred, y_true):
    """
    计算交叉熵损失
    
    参数:
    y_pred - 预测概率 (m x 10)
    y_true - 真实标签 (m,)
    
    返回:
    loss - 平均交叉熵损失
    """
    m = y_true.shape[0]
    
    # 创建one-hot编码的真实标签
    y_one_hot = np.zeros_like(y_pred)
    y_one_hot[np.arange(m), y_true] = 1
    
    # 计算交叉熵
    log_prob = np.log(np.clip(y_pred, 1e-15, 1 - 1e-15))
    loss = -np.sum(y_one_hot * log_prob) / m
    
    return loss

In [ ]:
# 实验要求5：交叉熵损失函数和梯度计算（修复版）
def backpropagation(cache, y_true, W2):
    """
    反向传播计算梯度（修复版）
    
    参数:
    cache - 前向传播中缓存的中间结果
    y_true - 真实标签
    W2 - 隐藏层到输出层的权重矩阵（用于梯度计算）
    
    返回:
    grads - 参数字典 {'dW1', 'db1', 'dW2', 'db2'}
    """
    m = y_true.shape[0]
    
    # 创建one-hot编码的真实标签
    y_one_hot = np.zeros_like(cache["A2"])
    y_one_hot[np.arange(m), y_true] = 1
    
    # 输出层梯度
    dZ2 = cache["A2"] - y_one_hot
    dW2 = cache["A1"].T.dot(dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m
    
    # 隐藏层梯度
    dA1 = dZ2.dot(W2.T)  # 修正：使用传递进来的W2参数，而不是从缓存获取
    dZ1 = dA1 * (cache["Z1"] > 0)  # ReLU导数
    dW1 = cache["X"].T.dot(dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m
    
    # 存储梯度
    grads = {
        "dW1": dW1, "db1": db1,
        "dW2": dW2, "db2": db2
    }
    
    return grads

In [ ]:
# 实验要求6：多层感知机训练程序（修复版）
def train_mlp(X_train, y_train, X_val, y_val, 
              hidden_size=128, 
              learning_rate=0.1, 
              epochs=200, 
              batch_size=64):
    """
    训练多层感知机（修复版）
    
    参数:
    X_train, y_train - 训练数据和标签
    X_val, y_val - 验证数据和标签
    hidden_size - 隐藏层大小 (默认128)
    learning_rate - 学习率 (默认0.1)
    epochs - 训练轮数 (默认200)
    batch_size - 批大小 (默认64)
    
    返回:
    model - 训练好的MLP模型
    history - 训练历史记录
    """
    n_features = X_train.shape[1]
    n_classes = len(np.unique(y_train))
    n_samples = X_train.shape[0]
    
    # 初始化模型
    model = MultiLayerPerceptron(n_features, hidden_size, n_classes)
    
    # 存储训练历史
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': []
    }
    
    # 训练循环
    for epoch in range(epochs):
        epoch_train_loss = 0
        epoch_train_acc = 0
        
        # 随机打乱数据
        indices = np.arange(n_samples)
        np.random.shuffle(indices)
        X_train_shuffled = X_train[indices]
        y_train_shuffled = y_train[indices]
        
        # 分批训练
        for i in range(0, n_samples, batch_size):
            # 获取当前批次
            X_batch = X_train_shuffled[i:i+batch_size]
            y_batch = y_train_shuffled[i:i+batch_size]
            
            # 前向传播
            y_pred, cache = model.forward(X_batch)
            
            # 计算损失
            loss = cross_entropy_loss(y_pred, y_batch)
            epoch_train_loss += loss * X_batch.shape[0]
            
            # 反向传播 - 传递当前权重W2
            grads = backpropagation(cache, y_batch, model.W2)
            
            # 参数更新
            model.W1 -= learning_rate * grads["dW1"]
            model.b1 -= learning_rate * grads["db1"]
            model.W2 -= learning_rate * grads["dW2"]
            model.b2 -= learning_rate * grads["db2"]
            
            # 计算批次准确率
            y_pred_labels = np.argmax(y_pred, axis=1)
            batch_acc = accuracy_score(y_batch, y_pred_labels)
            epoch_train_acc += batch_acc * X_batch.shape[0]
        
        # 计算平均训练损失和准确率
        epoch_train_loss /= n_samples
        epoch_train_acc /= n_samples
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        
        # 验证评估
        y_val_pred, _ = model.forward(X_val)
        val_loss = cross_entropy_loss(y_val_pred, y_val)
        y_val_pred_labels = np.argmax(y_val_pred, axis=1)
        val_acc = accuracy_score(y_val, y_val_pred_labels)
        
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # 每10轮打印进度
        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {epoch_train_loss:.4f}, "
                  f"Val Loss: {val_loss:.4f}, Train Acc: {epoch_train_acc:.4f}, "
                  f"Val Acc: {val_acc:.4f}")
    
    # 最终评估
    y_val_pred, _ = model.forward(X_val)
    y_val_pred_labels = np.argmax(y_val_pred, axis=1)
    final_val_acc = accuracy_score(y_val, y_val_pred_labels)
    print(f"\nTraining completed. Final validation accuracy: {final_val_acc:.4f}")
    
    return model, history

In [ ]:
# 实验要求7：分类正确率计算程序
def compute_accuracy(model, X, y):
    """
    计算模型在数据集上的分类正确率
    
    参数:
    model - 训练好的MLP模型
    X - 特征矩阵
    y - 真实标签
    
    返回:
    accuracy - 分类正确率
    y_pred - 预测标签
    """
    y_pred_proba, _ = model.forward(X)
    y_pred = np.argmax(y_pred_proba, axis=1)
    accuracy = accuracy_score(y, y_pred)
    return accuracy, y_pred

In [ ]:
# 6. 训练多层感知机
print("\n开始训练多层感知机...")
mlp_model, history = train_mlp(
    X_train, y_train, 
    X_test, y_test,  # 注意：这里我们使用测试集作为验证集（实际应用中应使用单独的验证集）
    hidden_size=128, 
    learning_rate=0.05,  # 稍微降低学习率以获得更稳定的训练
    epochs=50, 
    batch_size=256
)

In [ ]:
# 7. 计算训练集和测试集分类正确率
train_accuracy, _ = compute_accuracy(mlp_model, X_train, y_train)
test_accuracy, y_test_pred = compute_accuracy(mlp_model, X_test, y_test)
    
print("\n模型性能评估:")
print(f"训练集准确率: {train_accuracy:.4f}")
print(f"测试集准确率: {test_accuracy:.4f}")

In [ ]:
# 实验要求8：可视化训练结果
def plot_training_history(history):
    """
    可视化训练过程中的损失和准确率变化
    
    参数:
    history - 训练历史记录
    """
    epochs = len(history['train_loss'])
    
    plt.figure(figsize=(14, 5))
    
    # 绘制损失曲线
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], 'b-', label='Training Loss')
    plt.plot(history['val_loss'], 'r-', label='Validation Loss')
    plt.title('Loss Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    # 绘制准确率曲线
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], 'b-', label='Training Accuracy')
    plt.plot(history['val_acc'], 'r-', label='Validation Accuracy')
    plt.title('Accuracy Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 可视化混淆矩阵
def plot_confusion_matrix(y_true, y_pred, classes):
    """
    绘制混淆矩阵
    
    参数:
    y_true - 真实标签
    y_pred - 预测标签
    classes - 类别标签列表
    """
    from sklearn.metrics import confusion_matrix
    import seaborn as sns
    
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=classes, yticklabels=classes)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

In [ ]:
# 8. 可视化训练结果
plot_training_history(history)
    
# 额外：绘制混淆矩阵
plot_confusion_matrix(y_test, y_test_pred, classes=[str(i) for i in range(10)])